# Analyst Workbench

Este cuaderno te deja escoger un `ticker`, una `fecha`, un `agente` y una `peticion`.

El flujo hace esto:
- te muestra que data requiere el agente elegido
- corre automaticamente los prerequisitos necesarios
- ejecuta la etapa del agente con los modelos del repo
- genera una respuesta final orientada a tu peticion
- guarda artefactos en disco para que puedas inspeccionarlos

Importante:
- No expone chain-of-thought privado del modelo.
- Si falta data suficiente, la respuesta debe indicarlo explicitamente.

In [5]:
from pathlib import Path
import json
import os
import sys
from pprint import pprint
from IPython.display import Markdown, display

REPO_ROOT = Path(r'C:\Users\alemu\OneDrive\Documentos\TradingAgents')
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from analyst_access.workbench import describe_agent, list_agents, run_agent_request

TICKER = 'APTV'
TRADE_DATE = '2026-05-18'
AGENT_KEY = 'research_manager'
USER_REQUEST = 'Revisa si hay algún elemento en sus fundamentales que se haya degradado en los últimos 6 meses y que pueda afectar el precio de la acción en los próximos 3 a 12 meses.'

# Opcional: usar los defaults del repo dejando None.
QUICK_MODEL = 'gpt-5.4-mini'
DEEP_MODEL = 'gpt-5.5'
REASONING_EFFORT = 'medium'

OUTPUT_ROOT = 'analyst_access_sessions'


In [6]:
display(Markdown('## Agentes disponibles'))
for item in list_agents():
    print(f"- {item['agent_key']}: {item['display_name']}")
    print('  prereqs:', item['prerequisites'])
    for needed in item['required_data']:
        print('   *', needed)
    print()

## Agentes disponibles

- market_analyst: Market Analyst
  prereqs: []
   * OHLCV price history for the ticker
   * technical indicators such as moving averages, MACD, RSI, Bollinger bands, ATR, and VWMA
   * enough recent market data to support a directional market view

- social_media_analyst: Social Media Analyst
  prereqs: []
   * recent company-specific news and public discussion proxies
   * enough recent articles to infer sentiment and narrative pressure

- news_analyst: News Analyst
  prereqs: []
   * company-specific recent news
   * global and macro market news over the recent lookback window
   * enough macro context to explain possible impact on the ticker

- fundamentals_analyst: Fundamentals Analyst
  prereqs: []
   * company profile and high-level fundamentals
   * income statement data
   * balance sheet data
   * cash flow statement data

- bull_researcher: Bull Researcher
  prereqs: ['market_analyst', 'social_media_analyst', 'news_analyst', 'fundamentals_analyst']
   * market report
   * sen

In [7]:
spec = describe_agent(AGENT_KEY)
display(Markdown(f"## Agente seleccionado: `{AGENT_KEY}`"))
pprint(spec)

## Agente seleccionado: `research_manager`

{'agent_key': 'research_manager',
 'context_fields': ['investment_debate_state', 'investment_plan'],
 'display_name': 'Research Manager',
 'goal_hint': 'Synthesize the debate into a clear investment plan.',
 'kind': 'derived',
 'llm': 'deep',
 'prerequisites': ['market_analyst',
                   'social_media_analyst',
                   'news_analyst',
                   'fundamentals_analyst',
                   'bull_researcher',
                   'bear_researcher'],
 'required_data': ['bull and bear debate history',
                   'analyst reports from market, sentiment, news, and '
                   'fundamentals'],
 'stage_slug': '07_research_manager'}


In [8]:
result = run_agent_request(
    ticker=TICKER,
    trade_date=TRADE_DATE,
    agent_key=AGENT_KEY,
    user_request=USER_REQUEST,
    output_root=OUTPUT_ROOT,
    quick_model=QUICK_MODEL,
    deep_model=DEEP_MODEL,
    reasoning_effort=REASONING_EFFORT,
)

display(Markdown('## Resultado de la sesion'))
pprint({k: v for k, v in result.items() if k != 'answer'})
display(Markdown('## Respuesta del agente'))
display(Markdown(result['answer']))

## Resultado de la sesion

{'agent_key': 'research_manager',
 'display_name': 'Research Manager',
 'execution_plan': ['market_analyst',
                    'social_media_analyst',
                    'news_analyst',
                    'fundamentals_analyst',
                    'bull_researcher',
                    'bear_researcher',
                    'research_manager'],
 'output_dir': 'analyst_access_sessions\\APTV\\2026-05-18\\research_manager\\20260518_101526',
 'required_data': ['bull and bear debate history',
                   'analyst reports from market, sentiment, news, and '
                   'fundamentals'],
 'ticker': 'APTV',
 'trade_date': '2026-05-18',
 'user_request': 'Revisa si hay algún elemento en sus fundamentales que se '
                 'haya degradado en los últimos 6 meses y que pueda afectar el '
                 'precio de la acción en los próximos 3 a 12 meses.'}


## Respuesta del agente

Sí. Con la evidencia disponible, **sí hay deterioro fundamental relevante en APTV** que puede afectar negativamente la acción en los próximos **3 a 12 meses**. El punto central no es que la compañía esté “rota”, sino que la **calidad financiera se ha degradado**: peor conversión de caja, mayor capital de trabajo, más deuda y presión de márgenes.

## Conclusión ejecutiva

Mi lectura es **negativa / cautelosa** para APTV.

Aunque APTV todavía tiene ingresos, beneficios operativos, liquidez y exposición a áreas atractivas como ADAS, software y electrónica vehicular, los fundamentales recientes muestran señales claras de deterioro:

1. **Flujo de caja operativo pasó de +818M a -143M.**
2. **Flujo de caja libre pasó de +651M a -362M.**
3. **Cuentas por cobrar subieron a 4.297B.**
4. **Inventarios subieron a 2.746B.**
5. **Deuda total aumentó a aproximadamente 9.886B.**
6. **Deuda neta subió a aproximadamente 6.177B.**
7. **Hay presión en márgenes y menor calidad de las ganancias.**

Esto puede pesar sobre la acción durante los próximos trimestres, especialmente si el mercado concluye que el deterioro de caja no fue temporal.

---

## Elementos fundamentales que se han degradado

### 1. Conversión de caja: deterioro más importante

El dato más preocupante es el cambio en flujo de caja:

| Métrica | Antes | Último dato | Lectura |
|---|---:|---:|---|
| Flujo de caja operativo | +818M | -143M | Deterioro fuerte |
| Flujo de caja libre | +651M | -362M | Deterioro fuerte |

Este es el principal riesgo fundamental. APTV puede seguir reportando ingresos y EPS positivos, pero si no convierte esas ganancias en caja, el mercado tiende a castigar el múltiplo.

Para una empresa cíclica de autopartes, el flujo de caja libre es clave porque determina:

- capacidad de reducir deuda,
- flexibilidad ante una desaceleración automotriz,
- capacidad de financiar inversión,
- confianza en la calidad de las utilidades.

En este caso, el deterioro de caja es suficientemente grande como para afectar el precio de la acción en los próximos 3 a 12 meses.

---

### 2. Capital de trabajo: aumento de cuentas por cobrar e inventarios

La presión de caja parece venir en parte del capital de trabajo:

- **Cuentas por cobrar:** 4.297B  
- **Inventarios:** 2.746B  

Esto puede interpretarse de dos formas.

La visión optimista diría que es un problema temporal de timing. Si las cobranzas se normalizan y los inventarios se reducen, el flujo de caja podría recuperarse rápidamente.

Pero la visión más prudente es que este aumento puede señalar:

- menor eficiencia operativa,
- presión de demanda,
- acumulación de inventario,
- retrasos de cobro,
- menor poder de negociación con clientes OEM.

Hasta que se vea una reversión clara, este es un factor negativo para la acción.

---

### 3. Mayor deuda y menor flexibilidad financiera

Otro deterioro relevante es el balance:

| Métrica | Último dato |
|---|---:|
| Deuda total | 9.886B |
| Deuda neta | 6.177B |
| Caja y equivalentes | 3.173B |
| Current ratio | 2.11 |

La compañía no parece estar en estrés inmediato de liquidez, porque mantiene más de 3B en caja y un current ratio de 2.11. Pero el problema es la dirección: **la deuda sube justo cuando el flujo de caja libre se vuelve negativo**.

Esa combinación es problemática porque reduce el margen de error. Si los próximos trimestres muestran márgenes débiles o menor producción automotriz, el mercado puede exigir un mayor descuento por riesgo financiero.

---

### 4. Presión en márgenes

Los márgenes todavía no están colapsando, pero sí hay señales de presión:

- Margen bruto aproximado: **18.1%**
- Margen operativo aproximado: **8.7%**
- EBITDA normalizado: **691M**
- Ingreso operativo: **440M**

Estos niveles no indican una empresa en crisis, pero la preocupación es la dirección. Según el debate, hubo:

- presión de margen,
- menor ingreso operativo,
- debilidad regional,
- ingresos secuencialmente más débiles.

En un proveedor automotriz, pequeños deterioros de margen pueden tener impacto importante en la valoración, porque el negocio tiene sensibilidad a volumen, costos y ciclo.

---

## Factores que todavía sostienen el caso positivo

No todo es negativo. Hay elementos que evitan tratar a APTV como una compañía estructuralmente dañada:

- Ingresos del último trimestre: **5.086B**
- Ingreso neto: **189M**
- La compañía superó estimaciones:
  - EPS: **+5.30% vs consenso**
  - ingresos: **+1.27% vs consenso**
- Exposición a ADAS, software, electrónica vehicular y China.
- Caja de **3.173B**.
- Posible mejora de enfoque tras la separación de EDS.

Pero estos factores positivos no eliminan el riesgo principal: **la acción puede seguir bajo presión si el mercado percibe que la generación de caja se ha deteriorado de forma persistente.**

---

## Impacto probable en la acción en los próximos 3 a 12 meses

El deterioro fundamental más importante para el precio es la combinación de:

**flujo de caja libre negativo + aumento de deuda + presión de márgenes.**

Eso puede provocar:

1. **Compresión de múltiplos**  
   Aunque APTV reporte EPS positivos, el mercado podría asignarle un múltiplo menor si la caja no acompaña.

2. **Mayor sensibilidad a datos trimestrales**  
   Los próximos resultados serán críticos. Si no mejora el flujo de caja, la acción podría seguir cayendo.

3. **Menor apetito institucional**  
   Muchos inversores industriales/cíclicos evitan compañías con deterioro simultáneo de caja y deuda.

4. **Riesgo de revisión de expectativas**  
   Si la presión de márgenes persiste, el mercado podría reducir expectativas de beneficios futuros.

---

## Plan de inversión / seguimiento

Mi postura sería **vender o mantener infraponderado APTV** hasta ver evidencia de recuperación en caja.

### Señales necesarias para cambiar a una visión más positiva

Buscaría confirmar:

- flujo de caja operativo vuelve a positivo,
- flujo de caja libre vuelve a positivo,
- cuentas por cobrar dejan de crecer,
- inventarios se estabilizan o bajan,
- deuda neta se estabiliza o disminuye,
- margen operativo se mantiene o mejora,
- management mantiene o mejora guía.

Sin esas señales, el argumento de crecimiento en ADAS/software no es suficiente para compensar el deterioro financiero reciente.

### Niveles de riesgo

Con la acción ya castigada desde aproximadamente **88–89 hacia la zona de 54**, puede haber rebotes técnicos. Pero fundamentalmente, el riesgo sigue siendo bajista si el próximo trimestre confirma debilidad de caja.

Zona a vigilar:

- Primer riesgo bajista: **mediados/altos 40s**
- Si el flujo de caja negativo persiste: posible presión hacia **bajos 40s o menos**
- Señal de alivio: recuperación hacia **bajos 60s** acompañada de mejora real en caja, margen y deuda neta.

---

## Respuesta directa

Sí, hay degradación fundamental relevante en APTV en los últimos datos disponibles. Los elementos más preocupantes son:

1. **Deterioro fuerte del flujo de caja operativo.**
2. **Flujo de caja libre negativo.**
3. **Aumento de cuentas por cobrar e inventario.**
4. **Mayor deuda total y deuda neta.**
5. **Presión en márgenes y calidad de utilidades.**

Estos factores pueden afectar negativamente la acción en los próximos **3 a 12 meses**, salvo que la compañía demuestre rápidamente que el problema de caja fue temporal y que el capital de trabajo se normaliza.

## MISSING DATA:
- Estados financieros trimestrales completos de APTV de los últimos 6 meses para confirmar la evolución exacta trimestre a trimestre.
- Guía actualizada de management para ingresos, margen operativo, capex y flujo de caja libre 2026.
- Detalle por segmento tras la separación de EDS para evaluar qué parte del deterioro es estructural y qué parte es transitoria.
- Vencimientos de deuda, costo promedio de deuda y covenant headroom.
- Evolución de consenso de EPS, EBITDA y FCF para los próximos 4 trimestres.
- Datos actualizados de producción automotriz/OEM por región, especialmente Norteamérica, Europa y China.

In [ ]:
session_dir = Path(result['output_dir'])
display(Markdown(f'## Carpeta de salida\n`{session_dir}`'))

for path in sorted(session_dir.iterdir()):
    print(path.name)

In [ ]:
summary_path = session_dir / 'session_summary.json'
if summary_path.exists():
    display(Markdown('## Session summary'))
    pprint(json.loads(summary_path.read_text(encoding='utf-8')))

answer_path = session_dir / 'selected_answer.md'
if answer_path.exists():
    display(Markdown('## Respuesta guardada en disco'))
    display(Markdown(answer_path.read_text(encoding='utf-8')))

In [ ]:
display(Markdown('## Etapas ejecutadas y archivos generados'))
for path in sorted([p for p in session_dir.iterdir() if p.is_dir()]):
    print(f'[{path.name}]')
    for child in sorted(path.iterdir()):
        print('  -', child.name)
    print()